# 03 특징 엔지니어링

주간 type×family 패널에 lag/rolling 및 달력·외생 변수 피처를 생성합니다.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))
from utils.paths import DATA_PROCESSED

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
dfw = dfw.sort_values(['type','family','yearweek']).reset_index(drop=True)
dfw.head()



In [ ]:
# lag / rolling (주간)
for lag in [1, 2, 4, 8]:
    dfw[f'lag_{lag}'] = dfw.groupby(['type','family'])['sales'].shift(lag)
for w in [4, 8, 12]:
    dfw[f'roll_mean_{w}'] = dfw.groupby(['type','family'])['sales'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).mean())
    dfw[f'roll_std_{w}'] = dfw.groupby(['type','family'])['sales'].transform(lambda s: s.shift(1).rolling(w, min_periods=1).std())

dfw['zero_ratio_12'] = dfw.groupby(['type','family'])['sales'].transform(lambda s: (s.shift(1).rolling(12, min_periods=1) == 0).mean())
dfw.head()



In [ ]:
# 달력 피처
dfw['month'] = ((dfw['yearweek'] % 100) // 4 + 1).clip(1, 12)  # rough month proxy from week
# yearweek에서 연도
dfw['year_feat'] = dfw['yearweek'] // 100

feat_cols = [c for c in dfw.columns if c not in ['sales']]
print('feature columns:', len(feat_cols))
dfw[feat_cols].head()



In [ ]:
out = DATA_PROCESSED / 'df_weekly_features.parquet'
dfw.to_parquet(out, index=False)
print('저장:', out, '| shape:', dfw.shape)

